# Descifrado de imagen

Reversa exacta de `encrypt.ipynb`: primero deshace la permutacion del
automata celular (recupera la magnitud real por canal), reconstruye los
numeros complejos con la magnitud imaginaria y el mapa de signos guardados
sin cifrar, y por ultimo deshace la difusion con las matrices de Pauli
inversas.

Despues del descifrado principal hay tres secciones de analisis: sensibilidad
a la llave, robustez ante ruido/oclusion, y descifrado de imagenes planas
(negro/blanco) como caso de control.

In [ ]:
%pip install numpy pillow matplotlib tifffile

In [ ]:
import ast
import copy
import math
import random
from pathlib import Path

import numpy as np
import tifffile
from PIL import Image
import matplotlib.pyplot as plt

from difusion_pauli import (
    generar_matrices_4x4, matrices_encriptacion, descifrar_canal_pauli,
    bloques_a_canal, canal_2d_a_bloques, reconstruir_complejo,
)
from permutacion_automata import generar_indices_reglas, descifrar_canal_ca
from io_imagen import leer_metadatos, filas_a_matriz_pixeles, bits_pixel
from metricas import calcular_mse, calcular_psnr

In [ ]:
base = Path().resolve().parent
carpeta_resultados = base / "Resultados"

ruta_cifrado = carpeta_resultados / "Img Cifrada.tiff"
ruta_meta = carpeta_resultados / "cifrado_meta.txt"
ruta_salida = carpeta_resultados / "Img Decifrada.png"

## Leer la llave, los parametros y la imagen cifrada

In [ ]:
meta = leer_metadatos(ruta_meta)
key = ast.literal_eval(meta["Llave"])
x0 = float(meta["x0"])
iteraciones_usadas = int(meta["iteraciones_usadas"])

filas_R_cif = tifffile.imread(str(ruta_cifrado), key=1)
filas_G_cif = tifffile.imread(str(ruta_cifrado), key=2)
filas_B_cif = tifffile.imread(str(ruta_cifrado), key=3)

r_imag = tifffile.imread(str(ruta_cifrado), key=4).astype(np.float64)
g_imag = tifffile.imread(str(ruta_cifrado), key=5).astype(np.float64)
b_imag = tifffile.imread(str(ruta_cifrado), key=6).astype(np.float64)

r_signos = tifffile.imread(str(ruta_cifrado), key=7).astype(np.uint8)
g_signos = tifffile.imread(str(ruta_cifrado), key=8).astype(np.uint8)
b_signos = tifffile.imread(str(ruta_cifrado), key=9).astype(np.uint8)

h, w = r_imag.shape

if "num_bits" in meta:
    num_bits = int(meta["num_bits"])
else:
    # Compatibilidad con archivos cifrados antes de que num_bits se
    # guardara en cifrado_meta.txt: se reconstruye adivinando a partir
    # del ancho de fila con padding
    posibles_bits = bits_pixel(filas_R_cif.shape[1], w)
    num_bits = posibles_bits[0] if posibles_bits else 8

print(f"num_bits={num_bits}")

## Permutacion inversa (automata celular)

In [ ]:
reglas = generar_indices_reglas(x0, h, iteraciones_ignorar=iteraciones_usadas)

filas_R = descifrar_canal_ca(filas_R_cif, reglas)
filas_G = descifrar_canal_ca(filas_G_cif, reglas)
filas_B = descifrar_canal_ca(filas_B_cif, reglas)

R_int = filas_a_matriz_pixeles(filas_R, num_bits, w)
G_int = filas_a_matriz_pixeles(filas_G, num_bits, w)
B_int = filas_a_matriz_pixeles(filas_B, num_bits, w)

## Difusion inversa (matrices de Pauli)

In [ ]:
c_r_2d = reconstruir_complejo(R_int, r_imag, r_signos)
c_g_2d = reconstruir_complejo(G_int, g_imag, g_signos)
c_b_2d = reconstruir_complejo(B_int, b_imag, b_signos)

bloques_r_cif = canal_2d_a_bloques(c_r_2d)
bloques_g_cif = canal_2d_a_bloques(c_g_2d)
bloques_b_cif = canal_2d_a_bloques(c_b_2d)

matrices_rotadas = generar_matrices_4x4()
encriptacion = matrices_encriptacion(key, matrices_rotacion=matrices_rotadas)

p_r = descifrar_canal_pauli(bloques_r_cif, encriptacion)
p_g = descifrar_canal_pauli(bloques_g_cif, encriptacion)
p_b = descifrar_canal_pauli(bloques_b_cif, encriptacion)

R_dec = bloques_a_canal(p_r, h, w)
G_dec = bloques_a_canal(p_g, h, w)
B_dec = bloques_a_canal(p_b, h, w)

imagen_rgb = np.stack([R_dec, G_dec, B_dec], axis=-1)
imagen_decifrada = Image.fromarray(imagen_rgb, mode='RGB')
display(imagen_decifrada)
imagen_decifrada.save(str(ruta_salida))

## Sensibilidad a la llave (efecto avalancha)

`intentar_descifrar` repite todo el pipeline con una llave/semilla de
prueba y devuelve la imagen resultante, para comparar contra el descifrado
correcto. Antes esta funcion (y `pipeline_descifrado`, mas abajo) armaban la
imagen final con `np.abs(...)` sobre el resultado complejo; eso es la
magnitud, no la parte real que realmente representa el pixel, y no coincide
con lo que hace `bloques_a_canal` en el descifrado principal. Las dos ahora
llaman a `bloques_a_canal` para quedar consistentes con el flujo de arriba.

In [ ]:
def intentar_descifrar(test_x0, test_key, test_iter):
    test_reglas = generar_indices_reglas(test_x0, h, iteraciones_ignorar=test_iter)

    t_filas_R = descifrar_canal_ca(filas_R_cif, test_reglas)
    t_filas_G = descifrar_canal_ca(filas_G_cif, test_reglas)
    t_filas_B = descifrar_canal_ca(filas_B_cif, test_reglas)

    t_R_int = filas_a_matriz_pixeles(t_filas_R, num_bits, w)
    t_G_int = filas_a_matriz_pixeles(t_filas_G, num_bits, w)
    t_B_int = filas_a_matriz_pixeles(t_filas_B, num_bits, w)

    t_c_r_2d = reconstruir_complejo(t_R_int, r_imag, r_signos)
    t_c_g_2d = reconstruir_complejo(t_G_int, g_imag, g_signos)
    t_c_b_2d = reconstruir_complejo(t_B_int, b_imag, b_signos)

    t_bloques_r = canal_2d_a_bloques(t_c_r_2d)
    t_bloques_g = canal_2d_a_bloques(t_c_g_2d)
    t_bloques_b = canal_2d_a_bloques(t_c_b_2d)

    t_encriptacion = matrices_encriptacion(test_key, matrices_rotacion=matrices_rotadas)

    t_p_r = descifrar_canal_pauli(t_bloques_r, t_encriptacion)
    t_p_g = descifrar_canal_pauli(t_bloques_g, t_encriptacion)
    t_p_b = descifrar_canal_pauli(t_bloques_b, t_encriptacion)

    img_reconstruida = np.zeros((h, w, 3), dtype=np.uint8)
    img_reconstruida[:, :, 0] = bloques_a_canal(t_p_r, h, w)
    img_reconstruida[:, :, 1] = bloques_a_canal(t_p_g, h, w)
    img_reconstruida[:, :, 2] = bloques_a_canal(t_p_b, h, w)

    return img_reconstruida


img_correcta = intentar_descifrar(x0, key, iteraciones_usadas)

# Clave incorrecta: cambio minusculo en el caos (x0 + 1e-14)
x0_modificado = x0 + 1e-14
img_x0_erroneo = intentar_descifrar(x0_modificado, key, iteraciones_usadas)

# Clave incorrecta: un solo elemento de la matriz de Pauli
key_modificada = list(copy.deepcopy(key))
key_modificada[0] = (key_modificada[0] + 1) % 24
img_key_erronea = intentar_descifrar(x0, key_modificada, iteraciones_usadas)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_correcta); axes[0].set_title("Descifrado correcto\n(clave original)"); axes[0].axis("off")
axes[1].imshow(img_x0_erroneo); axes[1].set_title("x0 + 1e-14"); axes[1].axis("off")
axes[2].imshow(img_key_erronea); axes[2].set_title("1 elemento de Pauli modificado"); axes[2].axis("off")
plt.tight_layout()
plt.show()

print(f"MSE (original vs x0 modificado): {calcular_mse(img_correcta, img_x0_erroneo):.2f}")
print(f"MSE (original vs Pauli modificado): {calcular_mse(img_correcta, img_key_erronea):.2f}")

## Robustez ante ruido y oclusion

In [1]:
def agregar_ruido_sal_pimienta(matriz_cifrada, probabilidad):
    ruido_matriz = np.copy(matriz_cifrada)
    filas, columnas = ruido_matriz.shape
    num_ruido = int(probabilidad * filas * columnas)
    for _ in range(num_ruido):
        y = random.randint(0, filas - 1)
        x = random.randint(0, columnas - 1)
        ruido_matriz[y, x] = 0 if random.random() < 0.5 else 1
    return ruido_matriz


def aplicar_oclusion(matriz_cifrada, x_inicio, y_inicio, ancho, alto):
    matriz_ocluida = np.copy(matriz_cifrada)
    matriz_ocluida[y_inicio:y_inicio + alto, x_inicio:x_inicio + ancho] = 0
    return matriz_ocluida


def pipeline_descifrado(filas_r_mod, filas_g_mod, filas_b_mod):
    R_ca = descifrar_canal_ca(filas_r_mod, reglas)
    G_ca = descifrar_canal_ca(filas_g_mod, reglas)
    B_ca = descifrar_canal_ca(filas_b_mod, reglas)

    R_int_mod = filas_a_matriz_pixeles(R_ca, num_bits, w)
    G_int_mod = filas_a_matriz_pixeles(G_ca, num_bits, w)
    B_int_mod = filas_a_matriz_pixeles(B_ca, num_bits, w)

    c_r_2d_mod = reconstruir_complejo(R_int_mod, r_imag, r_signos)
    c_g_2d_mod = reconstruir_complejo(G_int_mod, g_imag, g_signos)
    c_b_2d_mod = reconstruir_complejo(B_int_mod, b_imag, b_signos)

    bloques_r_cif_mod = canal_2d_a_bloques(c_r_2d_mod)
    bloques_g_cif_mod = canal_2d_a_bloques(c_g_2d_mod)
    bloques_b_cif_mod = canal_2d_a_bloques(c_b_2d_mod)

    p_r_mod = descifrar_canal_pauli(bloques_r_cif_mod, encriptacion)
    p_g_mod = descifrar_canal_pauli(bloques_g_cif_mod, encriptacion)
    p_b_mod = descifrar_canal_pauli(bloques_b_cif_mod, encriptacion)

    img_reconstruida = np.zeros((h, w, 3), dtype=np.uint8)
    img_reconstruida[:, :, 0] = bloques_a_canal(p_r_mod, h, w)
    img_reconstruida[:, :, 1] = bloques_a_canal(p_g_mod, h, w)
    img_reconstruida[:, :, 2] = bloques_a_canal(p_b_mod, h, w)

    return img_reconstruida


print("--- Ruido sal y pimienta (5%) ---")
filas_R_ruido = agregar_ruido_sal_pimienta(filas_R_cif, 0.05)
filas_G_ruido = agregar_ruido_sal_pimienta(filas_G_cif, 0.05)
filas_B_ruido = agregar_ruido_sal_pimienta(filas_B_cif, 0.05)
img_descifrada_ruido = pipeline_descifrado(filas_R_ruido, filas_G_ruido, filas_B_ruido)

print("--- Oclusion (bloque 50x50) ---")
filas_R_oclusion = aplicar_oclusion(filas_R_cif, 10, 10, 50, 50)
filas_G_oclusion = aplicar_oclusion(filas_G_cif, 10, 10, 50, 50)
filas_B_oclusion = aplicar_oclusion(filas_B_cif, 10, 10, 50, 50)
img_descifrada_oclusion = pipeline_descifrado(filas_R_oclusion, filas_G_oclusion, filas_B_oclusion)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img_descifrada_ruido); axs[0].set_title("Reconstruccion con S&P 5%"); axs[0].axis('off')
axs[1].imshow(img_descifrada_oclusion); axs[1].set_title("Reconstruccion con oclusion"); axs[1].axis('off')
plt.show()

--- Ruido sal y pimienta (5%) ---


NameError: name 'filas_R_cif' is not defined

## Imagenes planas

Genera una imagen completamente negra y una completamente blanca como casos
de control. Hay que cifrarlas por separado con `encrypt.ipynb` (usando estas
dos como imagen de entrada) antes de poder analizarlas aqui.

In [ ]:
imagen_plana_negra = np.zeros((256, 256, 3), dtype=np.uint8)
imagen_plana_blanca = np.ones((256, 256, 3), dtype=np.uint8) * 255

Image.fromarray(imagen_plana_negra).save("plana_negra.png")
Image.fromarray(imagen_plana_blanca).save("plana_blanca.png")